# 04 - Label Engineering

In this notebook, we apply the KDIGO (Kidney Disease: Improving Global Outcomes) AKI criteria to our processed datasets to generate prospective labels for prediction. 

The KDIGO criteria define AKI as:
- An increase in serum creatinine by $\ge 0.3$ mg/dl within 48 hours, OR
- An increase in serum creatinine to $\ge 1.5$ times baseline within 7 days, OR
- Urine volume $< 0.5$ ml/kg/h for 6 hours.

We predict the occurrence of these criteria at prediction horizons of 6, 12, and 24 hours to create early warning signals.

In [ ]:
# Colab setup (uncomment if running in Google Colab)
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/sentinel-poc
# !pip install -r requirements.txt

In [ ]:
import os
import pandas as pd
import sys

# Ensure the src module is in the path
sys.path.append('..')
from src import labels

# Load processed datasets (train, val, test splits)
data_dir = '../data/processed/'
train_df = pd.read_parquet(os.path.join(data_dir, 'train.parquet'))
val_df = pd.read_parquet(os.path.join(data_dir, 'val.parquet'))
test_df = pd.read_parquet(os.path.join(data_dir, 'test.parquet'))

print(f"Train shape: {train_df.shape}")
print(f"Val shape: {val_df.shape}")
print(f"Test shape: {test_df.shape}")

In [ ]:
horizons = [6, 12, 24]

print("Labeling Train dataset...")
train_labeled = labels.label_dataset(train_df, horizons=horizons)

print("\nLabeling Validation dataset...")
val_labeled = labels.label_dataset(val_df, horizons=horizons)

print("\nLabeling Test dataset...")
test_labeled = labels.label_dataset(test_df, horizons=horizons)

In [ ]:
print("Train Dataset Label Distribution:")
labels.print_label_distribution(train_labeled, horizons)

print("\nValidation Dataset Label Distribution:")
labels.print_label_distribution(val_labeled, horizons)

print("\nTest Dataset Label Distribution:")
labels.print_label_distribution(test_labeled, horizons)

## Class Imbalance Discussion

As seen in the label distributions above, Sepsis-Associated AKI is a highly imbalanced outcome, typically occurring in a minority of clinical observations. Models trained on such imbalanced data tend to predict the majority class (non-AKI) at the expense of recall for the positive class (AKI).

To handle this during model training, we can employ strategies such as:
1. **`scale_pos_weight`**: Used in tree-based models like XGBoost and LightGBM to scale the gradient of the minority class proportionally.
2. **Class weights**: Specifying `class_weight='balanced'` in scikit-learn models like Logistic Regression or Random Forests to adjust weights inversely proportional to class frequencies.
3. **Threshold tuning**: Setting lower prediction thresholds optimizing for Area Under the Precision-Recall Curve (AUPRC) or F1-score rather than simple Accuracy.

In [ ]:
# Save labeled datasets back to data/processed/
train_labeled.to_parquet(os.path.join(data_dir, 'train_labeled.parquet'))
val_labeled.to_parquet(os.path.join(data_dir, 'val_labeled.parquet'))
test_labeled.to_parquet(os.path.join(data_dir, 'test_labeled.parquet'))

print("Labeled datasets saved successfully to", data_dir)